# Notebook 09 — Final Report

This notebook writes the project report from the tables and JSON files produced by the earlier notebooks. Every count and score in the report is read from those files at runtime. The report does not add a model run and does not treat a prediction or a topic term list as a verified fact about public opinion or government action.


## 1. Write the report from the saved outputs

Read the counts and scores from the earlier result files at runtime.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")

PREP = ROOT / "outputs" / "01_data_preparation" / "20261018_content_review"
ANN = ROOT / "outputs" / "03_annotation" / "20261018_annotation"
TOPIC = ROOT / "outputs" / "04_topic_modeling" / "20261019_topic_models"
MODEL = ROOT / "outputs" / "05_model_development" / "20261019_supervised"
PRED = ROOT / "outputs" / "06_results_predictions" / "20261019_predictions"
EVENTS = ROOT / "outputs" / "07_events_timeline" / "20261019_events"
FIG = ROOT / "outputs" / "08_results_visualization" / "20261019_figures"
RUN_ID = "20261019_report"
OUT = ROOT / "outputs" / "09_final_report" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

relation = json.loads((ANN / "annotation_relation.json").read_text(encoding="utf-8"))
topic_decision = json.loads((TOPIC / "topic_model_selection.json").read_text(encoding="utf-8"))
selected = json.loads((MODEL / "selected_models.json").read_text(encoding="utf-8"))
overlap = json.loads((PRED / "sample_overlap.json").read_text(encoding="utf-8"))
events = json.loads((EVENTS / "events_summary.json").read_text(encoding="utf-8"))
heldout = pd.read_csv(MODEL / "heldout_test_scores.csv")
validation = pd.read_csv(MODEL / "validation_scores.csv")
agreement = pd.read_csv(PRED / "refit_agreement.csv")
topic_terms = pd.read_csv(TOPIC / "selected_topic_terms.csv")
comparison = pd.read_csv(TOPIC / "topic_model_comparison.csv")
balance = pd.read_csv(ANN / "annotation_class_balance.csv")
timeline = pd.read_csv(EVENTS / "post_timeline.csv")
figure_index = pd.read_csv(FIG / "figure_index.csv")


def fmt(value):
    if pd.isna(value):
        return "not computed"
    return f"{float(value):.3f}"

def markdown_table(frame):
    columns = list(frame.columns)
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for _, row in frame.iterrows():
        lines.append("| " + " | ".join(str(row[column]) for column in columns) + " |")
    return "\n".join(lines)

heldout_view = heldout[[
    "task", "selected_model", "validation_macro_f1", "test_macro_f1", "test_weighted_f1", "test_accuracy"
]].copy()
for column in ["validation_macro_f1", "test_macro_f1", "test_weighted_f1", "test_accuracy"]:
    heldout_view[column] = heldout_view[column].map(fmt)
agreement_view = agreement[["task", "selected_model", "heldout_test_macro_f1", "refit_agreement_macro_f1", "refit_agreement_accuracy"]].copy()
for column in ["heldout_test_macro_f1", "refit_agreement_macro_f1", "refit_agreement_accuracy"]:
    agreement_view[column] = agreement_view[column].map(fmt)
topic_view = comparison[["family", "setting", "n_topics", "coherence_c_npmi", "topic_diversity", "coverage"]].copy()
topic_view["coherence_c_npmi"] = topic_view["coherence_c_npmi"].map(fmt)
topic_view["topic_diversity"] = topic_view["topic_diversity"].map(fmt)
topic_view["coverage"] = topic_view["coverage"].map(fmt)
term_view = topic_terms[["topic_id", "n_comments", "provisional_term_label"]].copy()
balance_view = balance.copy()
balance_view["largest_share_of_annotated"] = balance_view["largest_share_of_annotated"].map(fmt)

lines = []
add = lines.append
add("# Quantifying the Digital Policy Feedback Loop")
add("")
add("## Question and data")
add("")
add("The study asks how public Facebook comments on posts about Somalia's National Identification and Registration Authority can be described, classified, and placed next to inventoried institutional communications. The working corpus contains " + str(relation["raw_comments"]) + " comments under 19 source posts. The manually annotated subset contains " + str(relation["annotated_comments"]) + " comments, which is " + f"{relation['annotated_share_of_raw']:.1%}" + " of the corpus. Every annotated key matches a raw comment.")
add("")
add("`collection_date` is the time the comment file was collected. `publication_date_register` is the researcher-verified publication date of the source post. The two dates are not substitutes.")
add("")
add("## Annotation")
add("")
add(relation["decision"])
add("")
add("A written codebook is not in the project folder, and no second coder is recorded, so no agreement statistic is calculated. The four supervised tasks are `policy_relevance`, `feedback_type`, `topic`, and `sentiment`. `actor` is described below and is not one of those tasks. No class was merged.")
add("")
add(markdown_table(balance_view))
add("")
add("Shares use the 1,199 annotated comments as the denominator.")
add("")
add("## Unsupervised topics")
add("")
add("NMF, LDA, and BERTopic were fit on all " + str(topic_decision["comments_in_corpus"]) + " comments. Manual topic labels were not a training target. BERTopic used `paraphrase-multilingual-MiniLM-L12-v2`. That choice does not establish that the embedding model understands Somali.")
add("")
add(topic_decision["selection_rule"])
add("")
add("Selected model: " + topic_decision["selected_family"] + " (" + topic_decision["selected_setting"] + "). Coherence c_npmi: " + fmt(topic_decision["coherence_c_npmi"]) + ". Diversity: " + fmt(topic_decision["topic_diversity"]) + ". Coverage: " + fmt(topic_decision["coverage"]) + ". Unassigned comments: " + str(topic_decision["unassigned_comments"]) + ".")
add("")
add(markdown_table(topic_view))
add("")
add("The labels below are the model's own top terms. They are not validated names of policy themes. Several topics are led by frequent Somali function words because no stopword list was applied. The project does not contain a Somali stopword list, so one was not invented for this run.")
add("")
add(markdown_table(term_view))
add("")
add(topic_decision["language_limitation"])
add("")
add("## Supervised classification")
add("")
add("Each task compared five models on a stratified split of the 1,199 labeled comments: 70 percent train, 10 percent validation, and 20 percent test, with seed 42. The selected model is the one with the highest validation macro-F1. The test split was scored after selection and was not used to choose the model.")
add("")
add(markdown_table(heldout_view))
add("")
add("The TF-IDF models use letter tokens and word bigrams. They do not use an English stopword list, and the project does not contain a Somali stopword list. The fifth model is logistic regression on the same multilingual embeddings used for BERTopic. Its score does not show that those embeddings understand Somali.")
add("")
add("## Corpus predictions and comment comparison")
add("")
add("The selected model for each task was refit on all 1,199 labeled comments and then applied to all 5,292 comments. The refit has seen the labels that a held-out test is meant to hide, so the agreement below is not the performance result. The performance result remains the test macro-F1 in the table above.")
add("")
add(markdown_table(agreement_view))
add("")
add("The comment sample contains " + str(overlap["unique_record_keys"]) + " unique comments: " + str(overlap["annotated_sample"]) + " annotated, " + str(overlap["unlabeled_sample"]) + " unlabeled, and " + str(overlap["additional_predicted_sample"]) + " further comments. Overlap between the samples is " + str(overlap["overlap_between_samples"]) + ". " + overlap["note"])
add("")
add("Predicted labels are model output. They were not manually corrected.")
add("")
add("## Events and the feedback-loop claim")
add("")
add(events["decision"])
add("")
add("Posts classed as thematically related: " + str(events["thematically_related_posts"]) + ". Posts classed as temporally associated: " + str(events["temporally_associated_posts"]) + ". Posts classed as unlinked or unresolved: " + str(events["unlinked_posts"]) + ". Posts classed as directly linked: " + str(events["directly_linked_posts"]) + ".")
add("")
add("POST_09 shares the registered date of the inventoried SONNA item RESP_001, 2025-11-03. POST_01 and POST_04 are topic comparisons with named inventory rows. The register already says those comparisons do not establish causality. The five communications remain inventory rows. This study did not re-fetch the pages.")
add("")
add("POST_17 is used as a likely topic-level description of NIRA card issuance through the Somali Embassy in Uganda. The original Facebook caption was not recovered. POST_18 is used as a probable Eid al-Fitr greeting inferred from comments and from the publication date 2026-03-19. Its original caption was not recovered. Neither qualifier is evidence of an institutional response.")
add("")
add("POST_10 has a confirmed publication date of 2025-10-18. The policy claim in that post remains reported post content.")
add("")
add("## Figures")
add("")
add("The figures are saved from the tables above in `outputs/08_results_visualization/20261019_figures/figures/`.")
add("")
add(markdown_table(figure_index))
add("")
add("## Limits")
add("")
add("- The annotated rows are about 22 to 23 percent of every post. No sampling protocol is stored in the files, so the 1,199 rows are a manually labeled subset rather than a demonstrated random sample.")
add("- There is no codebook and no second coder.")
add("- Comments are short, mostly Somali, and written in informal Latin script. A few comments use Arabic script. Coherence and embedding distance are statistical descriptions, not a language-understanding test.")
add("- The largest classes are `relevant`, `complaint`, `other`, and `negative`. Macro-F1 gives equal weight to rare classes, including classes with few test rows.")
add("- Twenty-five annotation strings differ from the raw comment. Training used the raw working-corpus text. One difference is a redacted phone number. Excerpts in the sample redact phone-like strings.")
add("- Full-corpus predictions come from a model refit on every labeled row. They can be compared with the manual labels, and that comparison is not an independent test.")
add("- A shared topic, a nearby date, or a large predicted class is not evidence that an institution responded to a comment.")
add("")
add("## Files")
add("")
add("- Data preparation: `outputs/01_data_preparation/20261018_content_review/`")
add("- Exploratory analysis: `outputs/02_eda/20261018_eda/`")
add("- Annotation audit: `outputs/03_annotation/20261018_annotation/`")
add("- Topic models: `outputs/04_topic_modeling/20261019_topic_models/`")
add("- Supervised comparison: `outputs/05_model_development/20261019_supervised/`")
add("- Predictions: `outputs/06_results_predictions/20261019_predictions/`")
add("- Events: `outputs/07_events_timeline/20261019_events/`")
add("- Figures: `outputs/08_results_visualization/20261019_figures/`")
add("")
report = "\n".join(lines) + "\n"
(OUT / "FINAL_REPORT.md").write_text(report, encoding="utf-8")
(ROOT / "docs" / "FINAL_REPORT.md").write_text(report, encoding="utf-8")
print(report)


# Quantifying the Digital Policy Feedback Loop

## Question and data

The study asks how public Facebook comments on posts about Somalia's National Identification and Registration Authority can be described, classified, and placed next to inventoried institutional communications. The working corpus contains 5292 comments under 19 source posts. The manually annotated subset contains 1199 comments, which is 22.7% of the corpus. Every annotated key matches a raw comment.

`collection_date` is the time the comment file was collected. `publication_date_register` is the researcher-verified publication date of the source post. The two dates are not substitutes.

## Annotation

All 1,199 annotation keys match a raw comment. Twenty-five text differences are retained in the mismatch register and were not overwritten. One of those differences is a phone number redacted in the annotation file. The annotated share of each post is about 22 to 23 percent. No sampling protocol is stored in the project

### Interpretation

The report is written to `docs/FINAL_REPORT.md` and copied under `outputs/09_final_report/20261019_report/`. Every count in the generated text comes from a saved table or JSON file. The report states the selected LDA model, the four task winners, the test scores, the 30-comment sample, and the event split. It does not treat a prediction or a shared topic as a proven government response.
